<h1 style=\"text-align: center; font-size: 50px;\">😷 Register Model for COVID Movement Patterns with VAR (Vector Autoregression)</h1>
This notebook shows an visual data analysis of the effects of COVID-19 in two different cities: New York and London

## Notebook Overview
- Imports
- Configurations
- Preparing the Data
- Logging Model to MLflow
- Fetching the Latest Model Version from MLflow
- Loading the Model and Running Inference

## Imports

In [1]:
%%time

%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.
CPU times: user 59.3 ms, sys: 10.2 ms, total: 69.5 ms
Wall time: 1.51 s


In [2]:
# ------------------------ Data Manipulation ------------------------
import pandas as pd
import numpy as np

# ------------------------ System Utilities ------------------------
import warnings
import logging
from pathlib import Path
import os
import pickle
import time
import json
from typing import Any, Optional, Dict
import sys

# ------------------------ MLflow for Experiment Tracking and Model Management ------------------------
import mlflow
from mlflow import MlflowClient
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec, TensorSpec, ParamSchema, ParamSpec

# ------------------------ Statistical Analysis ------------------------
from statsmodels.tsa.stattools import adfuller

# ------------------------ New Models-from-Code Integration ------------------------
# Define the relative path to the 'src' directory (two levels up from current working directory)
src_path = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add 'src' directory to system path for module imports (e.g., utils)
if src_path not in sys.path:
    sys.path.append(src_path)

from src.utils import (
    load_config,)
from src.mlflow import Logger

## Configurations

In [3]:
# Suppress Python warnings
warnings.filterwarnings("ignore")

In [4]:
# Create logger
logger = logging.getLogger("cities_analysis_logger")
logger.setLevel(logging.INFO)
logger.propagate = False
logger.handlers.clear()

formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s", 
                              datefmt="%Y-%m-%d %H:%M:%S")  

stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

In [5]:
# ------------------------- Paths -------------------------
DATA_PATH = "/home/jovyan/datafabric/tutorial/"
ARTIFACTS_PATH = "../artifacts"
DEMO_FOLDER = "../demo"
CONFIG_PATH = "../configs/config.yaml"

# ------------------------ MLflow Integration ------------------------
EXPERIMENT_NAME = "Two_Cities_Experiment"
RUN_NAME = "Two_Cities_Run"
MODEL_NAME = "Two_Cities_Model"

In [6]:
start_time = time.time()  
logger.info('Notebook execution started.')

2025-09-02 17:16:40 - INFO - Notebook execution started.


## Verify Assets


In [7]:
# Check whether the Dataset file exists
is_dataset_available = Path(DATA_PATH).exists()

# Log the configuration status of the dataset
if is_dataset_available:
    logger.info("The Dataset is properly configured.")
else:
    logger.info(
        "The Dataset is not properly configured. Please create and download the required assets "
        "in your project on AI Studio."
    )

config = load_config(CONFIG_PATH)

print("✅ Configuration loaded successfully")


2025-09-02 17:16:40 - INFO - The Dataset is properly configured.


✅ Configuration loaded successfully


## Preparing the Data

### Acknowledgments:
I'd like to thank the original authors of these data sources!

| Data | Original Source |
| --- | --- |
| Mobility Data | [COVID-19 Community Mobility Reports](https://www.google.com/covid19/mobility/) |
| NYC Cases | [NYC Department of Health and Mental Hygiene](https://www1.nyc.gov/site/doh/index.page) |
| London Cases | [GOV.UK Coronavirus (COVID-19) in the UK](https://coronavirus.data.gov.uk/) |

In [8]:
source_folder = DATA_PATH
ny_mobility = pd.read_csv(f"{source_folder}/NewYork_mobility.csv")
ldn_mobility = pd.read_csv(f"{source_folder}London_mobility.csv")
ny_cases = pd.read_csv(f"{source_folder}daily_data_NewYork.csv")
ldn_cases = pd.read_csv(f"{source_folder}daily_data_London.csv")

In [9]:
def rename_mobility_cols(df: pd.DataFrame) -> pd.DataFrame:
    """
    Renames the and cleans the dataset columns.

    Parameters:
        df(pd.DataFrame): A Dataframe containing the data.

    Returns:
        pd.DataFrame: The cleaned dataframe with the renamed columns.
    """
    # Rename columns
    df = df.rename(columns={'country_region':'country'})
    df = df.rename(columns={'retail_and_recreation_percent_change_from_baseline':'retail'})
    df = df.rename(columns={'grocery_and_pharmacy_percent_change_from_baseline':'pharmacy'})
    df = df.rename(columns={'parks_percent_change_from_baseline':'parks'})
    df = df.rename(columns={'transit_stations_percent_change_from_baseline':'transit_station'})
    df = df.rename(columns={'workplaces_percent_change_from_baseline':'workplaces'})
    df = df.rename(columns={'residential_percent_change_from_baseline':'residential'})
    df.drop(['country_region_code','sub_region_1', 'sub_region_2', 'residential'], axis=1, inplace = True)
    return df


ny_mobility = ny_mobility.loc[ny_mobility['sub_region_2'] == "New York County"].reset_index(drop=True)
ldn_mobility = ldn_mobility.loc[ldn_mobility['sub_region_2'] == "City of London"].reset_index(drop=True)

ny_mobility = rename_mobility_cols(ny_mobility)
ldn_mobility = rename_mobility_cols(ldn_mobility)

mobility_features = ny_mobility.columns[6:]

| Mobility Features     | Description                                                                                                                           |
|-----------------|---------------------------------------------------------------------------------------------------------------------------------------|
| country          | Country Name                                                                         |
| metro_area       | Metropolitan area                                                                    |
| iso_3166_2_code  | Codes for the names of the principal subdivisions (e.g. provinces or states)         |
| census_fips_code | Census fips code                                                                     |
| place_id         | Place IDs uniquely identify a place in the Google Places database and on Google Maps |
| date             | Date                                                                                 |
| retail          | Mobility trends for places like restaurants, cafes, shopping centers, theme parks, museums, libraries, and movie theaters.            |
| pharmacy        | Mobility trends for places like grocery markets, food warehouses, farmers markets, specialty food shops, drug stores, and pharmacies. |
| parks           | Mobility trends for places like local parks, national parks, public beaches, marinas, dog parks, plazas, and public gardens.          |
| transit_station | Mobility trends for places like public transport hubs such as subway, bus, and train stations.                                        |
| workplaces      | Mobility trends for places of work.                                                                                                   |

In [10]:
ldn_mobility.head()

,country,metro_area,iso_3166_2_code,census_fips_code,place_id,date,retail,pharmacy,parks,transit_station,workplaces
0,United Kingdom,NaN,GB-LND,NaN,ChIJ4Y3fTlUDdkgR0Gbsoi2uDgQ,2020-02-15,-5.0,-9.0,-12.0,-11.0,NaN
1,United Kingdom,NaN,GB-LND,NaN,ChIJ4Y3fTlUDdkgR0Gbsoi2uDgQ,2020-02-16,-1.0,-21.0,-23.0,-13.0,NaN
2,United Kingdom,NaN,GB-LND,NaN,ChIJ4Y3fTlUDdkgR0Gbsoi2uDgQ,2020-02-17,-3.0,-2.0,4.0,-1.0,-4.0
3,United Kingdom,NaN,GB-LND,NaN,ChIJ4Y3fTlUDdkgR0Gbsoi2uDgQ,2020-02-18,-2.0,-2.0,-1.0,-2.0,-2.0
4,United Kingdom,NaN,GB-LND,NaN,ChIJ4Y3fTlUDdkgR0Gbsoi2uDgQ,2020-02-19,-7.0,-4.0,5.0,0.0,-4.0


Try it out yourself 🚀 : [Get the Address for a Place ID 🌎](https://developers.google.com/maps/documentation/javascript/examples/geocoding-place-id)

![](https://i.imgur.com/B69162k.png)

In [11]:
def rename_dailyData_cols(df: pd.DataFrame) -> pd.DataFrame:
    """
    Renames the daily data DataFrame columns to standardized names.

    Parameters:
        df (pd.DataFrame): A Dataframe containing daily COVID-19 statistics.

    Returns:
        pd.DataFrame: The Dataframe with the renamed columns.
    """
    mapping = {df.columns[0]:'date', df.columns[1]: 'case_count', df.columns[2]:'hospitalized_count', df.columns[3]: 'death_count'} 
    df = df.rename(columns = mapping)
    return df

ny_cases = ny_cases[['date_of_interest','CASE_COUNT','HOSPITALIZED_COUNT','DEATH_COUNT']]
ldn_cases = ldn_cases[['date','newCasesBySpecimenDate', 'newAdmissions', 'newDeaths28DaysByDeathDate']]

ny_cases = rename_dailyData_cols(ny_cases)
ldn_cases = rename_dailyData_cols(ldn_cases)

cases_features = ny_cases.columns[1:]

| Cases Features     | Description                    |
|--------------------|--------------------------------|
| date               | Date                           |
| case_count         | Number of daily cases recorded |
| hospitalized_count | Number of people hospitalized  |
| death_count        | Number of deaths recorded      |

In [12]:
ny_cases.head()

,date,case_count,hospitalized_count,death_count
0,02/29/2020,1,1,0
1,03/01/2020,0,1,0
2,03/02/2020,0,2,0
3,03/03/2020,1,7,0
4,03/04/2020,5,2,0


In [13]:
for df in ny_mobility, ldn_mobility, ny_cases, ldn_cases:
    df['date'] = df['date'].astype('datetime64[ns]')

In [14]:
def merge_data(mobility: pd.DataFrame, cases: pd.DataFrame) -> pd.DataFrame:
    """
    Merges mobility and case data on the 'date' column using an inner join.

    Parameters:
        mobility (pd.DataFrame): DataFrame containing mobility indicators.
        cases (pd.DataFrame): DataFrame containing COVID-19 case statistics.

    Returns:
        pd.DataFrame: Merged DataFrame containing both mobility and case data, aligned by date.
    """
    merged_df = pd.merge(mobility, cases, how='inner', on = 'date')
    return merged_df

# setting date as the index column
ny_df = merge_data(ny_mobility, ny_cases).set_index('date')
ldn_df = merge_data(ldn_mobility, ldn_cases).set_index('date')
ldn_df = ldn_df.iloc[:-2,:]

features = ny_df.columns[5:]

In [15]:
ldn_df.head()

,country,metro_area,iso_3166_2_code,census_fips_code,place_id,retail,pharmacy,parks,transit_station,workplaces,case_count,hospitalized_count,death_count
date,,,,,,,,,,,,,
2020-03-19,United Kingdom,NaN,GB-LND,NaN,ChIJ4Y3fTlUDdkgR0Gbsoi2uDgQ,-78.0,-47.0,-74.0,-70.0,-63.0,332,240,25
2020-03-20,United Kingdom,NaN,GB-LND,NaN,ChIJ4Y3fTlUDdkgR0Gbsoi2uDgQ,-82.0,-54.0,-77.0,-73.0,-64.0,424,272,46
2020-03-21,United Kingdom,NaN,GB-LND,NaN,ChIJ4Y3fTlUDdkgR0Gbsoi2uDgQ,-86.0,-53.0,-86.0,-78.0,NaN,348,311,46
2020-03-22,United Kingdom,NaN,GB-LND,NaN,ChIJ4Y3fTlUDdkgR0Gbsoi2uDgQ,-85.0,-67.0,-86.0,-81.0,NaN,439,335,53
2020-03-23,United Kingdom,NaN,GB-LND,NaN,ChIJ4Y3fTlUDdkgR0Gbsoi2uDgQ,-87.0,-70.0,-82.0,-79.0,-76.0,685,505,56


In [16]:
ny_df.head()

,country,metro_area,iso_3166_2_code,census_fips_code,place_id,retail,pharmacy,parks,transit_station,workplaces,case_count,hospitalized_count,death_count
date,,,,,,,,,,,,,
2020-02-29,United States,NaN,NaN,36061.0,ChIJOwE7_GTtwokRFq0uOwLSE9g,1.0,2.0,-2.0,-1.0,6.0,1,1,0
2020-03-01,United States,NaN,NaN,36061.0,ChIJOwE7_GTtwokRFq0uOwLSE9g,-3.0,2.0,-7.0,-3.0,1.0,0,1,0
2020-03-02,United States,NaN,NaN,36061.0,ChIJOwE7_GTtwokRFq0uOwLSE9g,3.0,6.0,17.0,-3.0,4.0,0,2,0
2020-03-03,United States,NaN,NaN,36061.0,ChIJOwE7_GTtwokRFq0uOwLSE9g,0.0,6.0,5.0,-1.0,3.0,1,7,0
2020-03-04,United States,NaN,NaN,36061.0,ChIJOwE7_GTtwokRFq0uOwLSE9g,3.0,9.0,13.0,-3.0,2.0,5,2,0


## Logging Model to MLflow

Reading the JSON with training parameters that were saved during training

In [17]:
artifacts_dir = ARTIFACTS_PATH
os.makedirs(artifacts_dir, exist_ok=True)

with open(f"{artifacts_dir}/training_metrics.json", "r") as metrics:
            metrics_dict = json.load(metrics)

In [18]:
# Define input and output schema for model signature
input_schema = Schema([
    ColSpec("string", "city"),
    ColSpec("long", "steps"),
])

output_schema = Schema([
    ColSpec("double", "retail_and_recreation_forecast"),
    ColSpec("double", "grocery_and_pharmacy_forecast"),
    ColSpec("double", "parks_forecast"),
    ColSpec("double", "transit_stations_forecast"),
    ColSpec("double", "workplaces_forecast"),
    ColSpec("double", "residential_forecast"),
    ColSpec("double", "cases_forecast"),
])

# Define model signature
signature = ModelSignature(inputs=input_schema, outputs=output_schema)

In [19]:
logger.info(f'Starting the experiment: {EXPERIMENT_NAME}')

mlflow.set_tracking_uri("/phoenix/mlflow")
# Set the MLflow experiment name
mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

# Start an MLflow run
with mlflow.start_run(run_name=RUN_NAME) as run:    
    
    # Registering the training metrics to mlflow
    mlflow.log_metrics(metrics_dict)
    
    # Print the artifact URI for reference
    logger.info(f"Run's Artifact URI: {run.info.artifact_uri}")
    
    # Log the model using new models-from-code approach
    
    Logger.log_model(
        signature=signature,
        artifact_path=MODEL_NAME,
        config_path=CONFIG_PATH,
        docs_path=ARTIFACTS_PATH,  # This contains the pickle files
        demo_folder=DEMO_FOLDER
    )

    # Register the logged model in MLflow Model Registry
    mlflow.register_model(
        model_uri=f"runs:/{run.info.run_id}/{MODEL_NAME}", 
        name=MODEL_NAME
    )

logger.info(f'Registered the model: {MODEL_NAME}')

2025-09-02 17:16:41 - INFO - Starting the experiment: Two_Cities_Experiment
2025-09-02 17:16:42 - INFO - Run's Artifact URI: /phoenix/mlflow/492114042865540463/ccb31468d17b425284bcb569daad4c87/artifacts
Registered model 'Two_Cities_Model' already exists. Creating a new version of this model...
2025/09/02 17:16:59 WARNING mlflow.tracking._model_registry.fluent: Run with id ccb31468d17b425284bcb569daad4c87 has no artifacts at artifact path 'Two_Cities_Model', registering model based on models:/m-84ddd688801f4fa9a21aafe2c5d33419 instead
Created version '8' of model 'Two_Cities_Model'.
2025-09-02 17:17:07 - INFO - Registered the model: Two_Cities_Model


## Fetching the Latest Model Version from MLflow

In [20]:
# Initialize the MLflow client
client = MlflowClient()

# Retrieve the latest version of the model
model_metadata = client.get_latest_versions(MODEL_NAME, stages=["None"])
latest_model_version = model_metadata[0].version  # Extract the latest model version

# Fetch model information, including its signature
model_info = mlflow.models.get_model_info(f"models:/{MODEL_NAME}/{latest_model_version}")

# Print the latest model version and its signature
print(f"Latest Model Version: {latest_model_version}")
print(f"Model Signature: {model_info.signature}")

Latest Model Version: 8
Model Signature: inputs: 
  ['city': string (required), 'steps': long (required)]
outputs: 
  ['retail_and_recreation_forecast': double (required), 'grocery_and_pharmacy_forecast': double (required), 'parks_forecast': double (required), 'transit_stations_forecast': double (required), 'workplaces_forecast': double (required), 'residential_forecast': double (required), 'cases_forecast': double (required)]
params: 
  None



## Loading the Model and Running Inference

In [21]:
model = mlflow.pyfunc.load_model(model_uri=f"models:/{MODEL_NAME}/{latest_model_version}")
# Make predictions for New York
ny_prediction = model.predict({
    "city": ["New York"],
    "steps": [3]  # Forecast for the next 3 days
})

# Make predictions for London
ldn_prediction = model.predict({
    "city": ["London"],
    "steps": [3]  # Forecast for the next 3 days
})

print(f"New York: {ny_prediction}" )
print("\n")
print(f"London: {ldn_prediction}")

New York:                             retail_forecast  pharmacy_forecast  \
2025-09-02 17:17:20.819225       -33.728463         -30.846299   
2025-09-03 17:17:20.819225       -38.215550         -30.564491   
2025-09-04 17:17:20.819225       -35.413120         -22.604498   

                            parks_forecast  transit_station_forecast  \
2025-09-02 17:17:20.819225       10.141208                -22.428500   
2025-09-03 17:17:20.819225       -4.275643                -39.144546   
2025-09-04 17:17:20.819225       11.427796                -30.944809   

                            workplaces_forecast  case_count_forecast  \
2025-09-02 17:17:20.819225           -22.994693          2408.753695   
2025-09-03 17:17:20.819225           -56.921910          4896.935950   
2025-09-04 17:17:20.819225           -40.361528          4383.142165   

                            hospitalized_count_forecast  death_count_forecast  
2025-09-02 17:17:20.819225                    98.599755            

In [22]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")
logger.info("✅ Notebook execution completed successfully.")

2025-09-02 17:17:20 - INFO - ⏱️ Total execution time: 0m 40.35s
2025-09-02 17:17:20 - INFO - ✅ Notebook execution completed successfully.


Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).